# LiNGAM 归因分析与三列等高桑基图（12个特征版本）

本 notebook 已按你的新要求完成以下修改：

- 新增 `TAS` 与 `HUSS`
- 忽略 `TASmax` 与 `TASmin`
- 三类因素调整为 `Anthropogenic activities`、`Climate change`、`Hydrogeology`
- 颜色调整为 `Reduces disaster = #AED3E6`、`Promotes disaster = #CB4942`
- 桑基图三列总高度保持一致

In [ ]:
# -*- coding: utf-8 -*-
"""
LiNGAM 归因分析 + 三列等高桑基图（12个特征版本）
更新内容：
1. 新增 TAS 与 HUSS，忽略 TASmax 与 TASmin
2. 三类因素改为：
   - Anthropogenic activities：UF；IP；PT；WTD；LAI
   - Climate change：PR；TAS；HUSS
   - Hydrogeology：DK；DB；DF；HDS
3. 颜色修改为：
   - Reduces disaster：#AED3E6
   - Promotes disaster：#CB4942
4. 三列总高度保持一致
5. 本次进一步修改：
   - 所有填充透明度改为 0（即不透明显示）
   - 第二列 Factor group 的三个因素间距固定约 2 mm
   - 第三列的彩色条带总高度与节点方块一致
"""

import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from lingam import DirectLiNGAM
from lingam.utils import make_prior_knowledge


def build_sankey_svg_equalheight(
    links_df: pd.DataFrame,
    out_svg: str,
    type_order: list[str],
    width_mm: float = 180.0,
    height_mm: float = 70.0,
    font_size_equiv_mm: float = 3.175,  # 约等于 9 pt
    margin_top: float = 12.0,
    margin_bottom: float = 6.0,
    margin_left: float = 18.0,
    margin_right: float = 10.0,
    node_w: float = 2.2,
    min_gap: float = 1.0,
    type_gap_mm: float = 2.0,
):
    """
    生成三列总高度一致的 SVG 桑基图：
    Feature -> Factor group -> Disaster

    本版本额外满足：
    1. 中间列 Factor group 的节点间距固定约 2 mm
    2. 第三列 Disaster 的彩色条带总高度与节点方块完全一致
    3. 所有填充不使用透明度
    """

    COL_REDUCE = "#AED3E6"
    COL_PROMOTE = "#CB4942"
    COL_NODE = "#B0B0B0"

    f2t = links_df[links_df["link_level"] == "feature_to_type"].copy()
    t2d = links_df[links_df["link_level"] == "type_to_disaster"].copy()

    features = list(dict.fromkeys(f2t["source"].tolist()))
    types = list(dict.fromkeys(f2t["target"].tolist()))
    types = [t for t in type_order if t in types] + [t for t in types if t not in type_order]

    f_tot = f2t.groupby("source")["value"].sum().reindex(features).fillna(0.0)
    features = sorted(features, key=lambda k: (-float(f_tot.get(k, 0.0)), k))
    f_tot = f_tot.reindex(features)

    t_tot = f2t.groupby("target")["value"].sum().reindex(types).fillna(0.0)
    total_flow = float(f2t["value"].sum())
    if total_flow <= 0:
        raise ValueError("总流量为 0，无法绘制桑基图。")

    node_area_h = height_mm - margin_top - margin_bottom

    # 第一列：按真实流量高度 + 自动间距铺满整列
    max_nodes = max(len(features), 1)
    total_flow_h_feature = node_area_h - min_gap * max(max_nodes - 1, 0)
    if total_flow_h_feature <= 0:
        raise ValueError("绘图区高度不足，请增大图高或减小 min_gap。")
    scale_feature = total_flow_h_feature / total_flow

    def stack_positions_feature(names, totals):
        flow_heights = [float(totals.get(n, 0.0)) * scale_feature for n in names]
        flow_sum = sum(flow_heights)
        n = len(names)
        positions = {}
        if n == 0:
            return positions
        if n == 1:
            extra_top = (node_area_h - flow_sum) / 2.0
            y0 = margin_top + extra_top
            positions[names[0]] = (y0, y0 + flow_heights[0])
            return positions
        gap = (node_area_h - flow_sum) / (n - 1)
        y = margin_top
        for name, h in zip(names, flow_heights):
            positions[name] = (y, y + h)
            y += h + gap
        return positions

    posF = stack_positions_feature(features, f_tot)

    # 第二列：固定约 2 mm 间距，节点本体按比例拉伸后铺满整列
    n_types = len(types)
    total_gap_type = type_gap_mm * max(n_types - 1, 0)
    usable_type_h = node_area_h - total_gap_type
    if usable_type_h <= 0:
        raise ValueError("中间列可用高度不足，请减小 type_gap_mm 或增大图高。")
    scale_type_node = usable_type_h / total_flow

    posT = {}
    y = margin_top
    for t in types:
        h = float(t_tot.get(t, 0.0)) * scale_type_node
        posT[t] = (y, y + h)
        y += h + type_gap_mm

    # 第三列：节点与彩色条带都铺满整列
    posD = {"Disaster": (margin_top, margin_top + node_area_h)}
    scale_disaster_node = node_area_h / total_flow

    xF = margin_left + 8.0
    xD = width_mm - margin_right - 30.0
    xT = (xF + xD) / 2.0

    # 第二列内部的正/负子带
    t_sign_tot = f2t.groupby(["target", "effect_sign"])["value"].sum()
    t_pos = {t: float(t_sign_tot.get((t, "pos"), 0.0)) for t in types}

    t_band_start = {}
    for t in types:
        y0, y1 = posT[t]
        pos_h = t_pos[t] * scale_type_node
        t_band_start[t] = {"pos": y0, "neg": y0 + pos_h}

    # 第三列内部的正/负子带，直接占满方块
    d_pos = float(t2d.loc[t2d["effect_sign"] == "pos", "value"].sum())
    d_band_start = {"pos": margin_top, "neg": margin_top + d_pos * scale_disaster_node}

    f_out_offset = {k: posF[k][0] for k in features}
    t_in_offset = {t: {"pos": t_band_start[t]["pos"], "neg": t_band_start[t]["neg"]} for t in types}
    t_out_offset = {t: {"pos": t_band_start[t]["pos"], "neg": t_band_start[t]["neg"]} for t in types}
    d_in_offset = {"pos": d_band_start["pos"], "neg": d_band_start["neg"]}

    sign_order = {"pos": 0, "neg": 1}
    f2t["tgt_y"] = f2t["target"].map(lambda t: posT[t][0])
    f2t["sord"] = f2t["effect_sign"].map(lambda s: sign_order.get(str(s), 9))
    f2t["src_y"] = f2t["source"].map(lambda s: posF[s][0])
    f2t = f2t.sort_values(["tgt_y", "sord", "src_y"]).drop(columns=["tgt_y", "sord", "src_y"])

    t2d["src_y"] = t2d["source"].map(lambda s: posT[s][0])
    t2d["sord"] = t2d["effect_sign"].map(lambda s: sign_order.get(str(s), 9))
    t2d = t2d.sort_values(["src_y", "sord"]).drop(columns=["src_y", "sord"])

    def ribbon_path(x0, x1, y0a, y0b, y1a, y1b, curv=0.55):
        dx = (x1 - x0) * curv
        return (
            f"M {x0:.3f} {y0a:.3f} "
            f"C {x0+dx:.3f} {y0a:.3f}, {x1-dx:.3f} {y1a:.3f}, {x1:.3f} {y1a:.3f} "
            f"L {x1:.3f} {y1b:.3f} "
            f"C {x1-dx:.3f} {y1b:.3f}, {x0+dx:.3f} {y0b:.3f}, {x0:.3f} {y0b:.3f} "
            f"Z"
        )

    def text_style():
        return f'style="font-size:{font_size_equiv_mm}; font-family:\'Times New Roman\', Times, serif;"'

    svg = []
    svg.append(
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width_mm}mm" height="{height_mm}mm" viewBox="0 0 {width_mm} {height_mm}">'
    )
    svg.append('<rect x="0" y="0" width="100%" height="100%" fill="white"/>')
    svg.append(
        '<style><![CDATA['
        f'text{{font-family:"Times New Roman", Times, serif;font-size:{font_size_equiv_mm};fill:#000;}}'
        '.node{stroke:#000;stroke-width:0.18;}'
        '.link{stroke:none;}'
        ']]></style>'
    )

    # 左 -> 中：允许厚度渐变，使中间列在固定间距下仍铺满总高度
    for _, r in f2t.iterrows():
        s = r["source"]
        t = r["target"]
        v = float(r["value"])
        sign = str(r["effect_sign"])

        thick0 = v * scale_feature
        thick1 = v * scale_type_node

        y0a = f_out_offset[s]
        y0b = y0a + thick0
        f_out_offset[s] = y0b

        y1a = t_in_offset[t][sign]
        y1b = y1a + thick1
        t_in_offset[t][sign] = y1b

        fill = COL_PROMOTE if sign == "pos" else COL_REDUCE
        svg.append(
            f'<path class="link" d="{ribbon_path(xF+node_w, xT, y0a, y0b, y1a, y1b)}" fill="{fill}" fill-opacity="1"/>'
        )

    # 中 -> 右：右侧条带总高度与方块完全一致
    for _, r in t2d.iterrows():
        t = r["source"]
        v = float(r["value"])
        sign = str(r["effect_sign"])

        thick0 = v * scale_type_node
        thick1 = v * scale_disaster_node

        y0a = t_out_offset[t][sign]
        y0b = y0a + thick0
        t_out_offset[t][sign] = y0b

        y1a = d_in_offset[sign]
        y1b = y1a + thick1
        d_in_offset[sign] = y1b

        fill = COL_PROMOTE if sign == "pos" else COL_REDUCE
        svg.append(
            f'<path class="link" d="{ribbon_path(xT+node_w, xD, y0a, y0b, y1a, y1b)}" fill="{fill}" fill-opacity="1"/>'
        )

    def draw_layer(nodes, pos, x, label_side):
        for n in nodes:
            y0, y1 = pos[n]
            h = max(0.2, y1 - y0)
            svg.append(
                f'<rect class="node" x="{x:.3f}" y="{y0:.3f}" width="{node_w:.3f}" height="{h:.3f}" fill="{COL_NODE}" fill-opacity="1"/>'
            )
            yc = (y0 + y1) / 2.0
            if label_side == "left":
                tx, anchor = x - 1.2, "end"
            else:
                tx, anchor = x + node_w + 1.2, "start"
            label = str(n).replace("&", "&amp;")
            svg.append(
                f'<text x="{tx:.3f}" y="{yc:.3f}" text-anchor="{anchor}" dominant-baseline="middle" {text_style()}>{label}</text>'
            )

    draw_layer(features, posF, xF, "left")
    draw_layer(types, posT, xT, "right")

    d0, d1 = posD["Disaster"]
    svg.append(
        f'<rect class="node" x="{xD:.3f}" y="{d0:.3f}" width="{node_w:.3f}" height="{(d1-d0):.3f}" fill="{COL_NODE}" fill-opacity="1"/>'
    )
    svg.append(
        f'<text x="{(xD+node_w+1.2):.3f}" y="{((d0+d1)/2.0):.3f}" text-anchor="start" dominant-baseline="middle" {text_style()}>Disaster</text>'
    )

    header_y = 4.5
    svg.append(f'<text x="{xF:.3f}" y="{header_y:.3f}" text-anchor="start" {text_style()}>Feature</text>')
    svg.append(f'<text x="{xT:.3f}" y="{header_y:.3f}" text-anchor="start" {text_style()}>Factor group</text>')
    svg.append(f'<text x="{xD:.3f}" y="{header_y:.3f}" text-anchor="start" {text_style()}>Disaster</text>')

    lx, ly, box = 2.0, 7.0, 2.2
    svg.append(
        f'<rect x="{lx:.3f}" y="{ly:.3f}" width="{box:.3f}" height="{box:.3f}" fill="{COL_PROMOTE}" fill-opacity="1" stroke="#000" stroke-width="0.18"/>'
    )
    svg.append(
        f'<text x="{(lx+box+1.2):.3f}" y="{(ly+box-0.3):.3f}" text-anchor="start" {text_style()}>Promotes disaster</text>'
    )
    ly2 = ly + 3.4
    svg.append(
        f'<rect x="{lx:.3f}" y="{ly2:.3f}" width="{box:.3f}" height="{box:.3f}" fill="{COL_REDUCE}" fill-opacity="1" stroke="#000" stroke-width="0.18"/>'
    )
    svg.append(
        f'<text x="{(lx+box+1.2):.3f}" y="{(ly2+box-0.3):.3f}" text-anchor="start" {text_style()}>Reduces disaster</text>'
    )

    svg.append("</svg>")

    with open(out_svg, "w", encoding="utf-8") as f:
        f.write("\n".join(svg))

    print(f"[OK] SVG 已保存：{out_svg}")


In [ ]:
# =========================
# 1) 路径与输出目录
# =========================
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_path = os.path.join(base_path, "data")
path_name = "Positive_Negative_balanced"
out_dir = os.path.join("/mnt/data", "lingam_updated_12vars")
os.makedirs(out_dir, exist_ok=True)

# 优先读取原路径，若不存在则回退到当前会话上传文件
df_path = os.path.join(
    data_path,
    "Extracted_HAVE_future",
    "Positive_Negative_balanced",
    "AllFeatures_Positive_Negative_balanced_25366_ssp1_cleaned.csv",
)
fallback_path = "/mnt/data/AllFeatures_Positive_Negative_balanced_25366_ssp1_cleaned.csv"
if (not os.path.exists(df_path)) and os.path.exists(fallback_path):
    print(f"[INFO] 未找到原始路径，改用：{fallback_path}")
    df_path = fallback_path

df = pd.read_csv(df_path)
print(f"[INFO] 读取数据：{df_path}")
print(f"[INFO] 数据形状：{df.shape}")

# =========================
# 2) 特征映射（新增 TAS / HUSS，忽略 TASmax / TASmin）
# =========================
feature_map = {
    "Distance_to_karst": "DK",
    "Depth_to_Bedrock": "DB",
    "Distance_to_Fault_m": "DF",
    "UrbanFrac_hist_2000_2010_2020": "UF",
    "PopTotal_hist_2000_2010_2020": "PT",
    "ImperviousIndex_hist_2000_2010_2020": "IP",
    "LAI_hist_2000_2010_2020": "LAI",
    "Precip_hist_2000_2010_2020": "PR",
    "Tas_hist_2000_2010_2020": "TAS",
    "Huss_hist_2000_2010_2020": "HUSS",
    "WTD_hist_2000_2010_2020": "WTD",
    "HDS_hist_2000_2010_2020": "HDS",
}
target_col = "Disaster"

type_members = {
    "Anthropogenic activities": {"UF", "IP", "PT", "WTD", "LAI"},
    "Climate change": {"PR", "TAS", "HUSS"},
    "Hydrogeology": {"DK", "DB", "DF", "HDS"},
}
type_order = ["Anthropogenic activities", "Climate change", "Hydrogeology"]

# code -> group
code_to_types = {}
for _, code in feature_map.items():
    groups = [g for g, codes in type_members.items() if code in codes]
    if not groups:
        groups = ["Unclassified"]
    code_to_types[code] = groups

# =========================
# 3) 数据清洗
# =========================
need_cols = list(feature_map.keys()) + [target_col]
missing = sorted(list(set(need_cols) - set(df.columns)))
if missing:
    raise ValueError(f"数据缺少列：{missing}")

df_sub = df[need_cols].copy().replace([np.inf, -np.inf], np.nan)
for c in need_cols:
    df_sub[c] = pd.to_numeric(df_sub[c], errors="coerce")
df_sub = df_sub.dropna(axis=0)
print(f"[INFO] dropna 后形状：{df_sub.shape}")

var = df_sub.var(axis=0)
keep_cols = [c for c in df_sub.columns if var.get(c, 0.0) > 1e-12]
if target_col not in keep_cols:
    raise ValueError("Disaster 列方差为 0，LiNGAM 无法运行。")
df_sub = df_sub[keep_cols]
print(f"[INFO] 去除常数列后列数：{len(df_sub.columns)}")

# =========================
# 4) LiNGAM 拟合
# =========================
cols_order = list(df_sub.columns)
X = df_sub[cols_order].values
Xz = StandardScaler().fit_transform(X)

dis_idx = cols_order.index(target_col)
pk = make_prior_knowledge(n_variables=len(cols_order), sink_variables=[dis_idx])

model = DirectLiNGAM(random_state=0, prior_knowledge=pk)
model.fit(Xz)

# adjacency: B[i, j] 表示 j -> i 的直接效应
B = model.adjacency_matrix_
W = np.linalg.inv(np.eye(B.shape[0]) - B)
total_to_disaster = W[dis_idx, :].copy()
total_to_disaster[dis_idx] = 0.0

# =========================
# 5) 保存每个特征到 Disaster 的总效应
# =========================
rows = []
for full, code in feature_map.items():
    if full not in cols_order:
        continue
    j = cols_order.index(full)
    te = float(total_to_disaster[j])
    rows.append({
        "feature_full": full,
        "feature_code": code,
        "total_effect": te,
        "abs_total_effect": abs(te),
        "sign": "pos" if te > 0 else ("neg" if te < 0 else "zero"),
        "factor_group": "; ".join(code_to_types.get(code, ["Unclassified"])),
    })

effects_df = pd.DataFrame(rows).sort_values("abs_total_effect", ascending=False)
effects_out = os.path.join(out_dir, "lingam_total_effects_to_Disaster_12vars.csv")
effects_df.to_csv(effects_out, index=False, encoding="utf-8-sig")
print(f"[OK] 总效应结果已保存：{effects_out}")

effects_df

In [ ]:
# =========================
# 6) 生成桑基图 links
# =========================
min_abs_effect = 1e-6
links = []
type_pos, type_neg = {}, {}

for _, r in effects_df.iterrows():
    code = r["feature_code"]
    full = r["feature_full"]
    te = float(r["total_effect"])
    mag = abs(te)

    if mag < min_abs_effect:
        continue

    sign = "pos" if te > 0 else ("neg" if te < 0 else "zero")
    if sign == "zero":
        continue

    groups = code_to_types.get(code, ["Unclassified"])
    share = mag / len(groups)

    for g in groups:
        links.append({
            "link_level": "feature_to_type",
            "source": code,
            "target": g,
            "value": share,
            "feature_full": full,
            "feature_code": code,
            "feature_type_en": g,
            "effect_sign": sign,
            "total_effect": te,
        })
        if sign == "pos":
            type_pos[g] = type_pos.get(g, 0.0) + share
        else:
            type_neg[g] = type_neg.get(g, 0.0) + share

for g in type_order:
    if type_pos.get(g, 0.0) >= min_abs_effect:
        links.append({
            "link_level": "type_to_disaster",
            "source": g,
            "target": "Disaster",
            "value": type_pos[g],
            "effect_sign": "pos",
        })
    if type_neg.get(g, 0.0) >= min_abs_effect:
        links.append({
            "link_level": "type_to_disaster",
            "source": g,
            "target": "Disaster",
            "value": type_neg[g],
            "effect_sign": "neg",
        })

links_df = pd.DataFrame(links)
links_out = os.path.join(out_dir, "sankey_lingam_links_disaster_color_12vars.csv")
links_df.to_csv(links_out, index=False, encoding="utf-8-sig")
print(f"[OK] links 已保存：{links_out}")

# =========================
# 7) 输出 SVG
# =========================
svg_out = os.path.join(out_dir, "sankey_lingam_12vars_equalheight_v2.svg")
build_sankey_svg_equalheight(
    links_df=links_df,
    out_svg=svg_out,
    type_order=type_order,
    width_mm=180.0,
    height_mm=70.0,
    font_size_equiv_mm=3.175,
    min_gap=1.0,
    type_gap_mm=2.0,
)

print("[DONE]")
print(f"[RESULT] Notebook 输出目录：{out_dir}")